# Forensics: what does the shop's hidden model reward? (H1–H11)

Questions from the plan (`docs/memory/06_plan_ganador.md` §3). All logic lives in `rtrec/data/forensics.py`;
this notebook only runs it and shows the tables, so **it reruns unchanged on the real dump**:

```bash
python scripts/download_history.py --out data --live      # real data (needs the key)
python -m sim.dump --out data --live-minutes 120          # or synthetic data
RTREC_DATA=data jupyter nbconvert --to notebook --execute notebooks/forensics.ipynb
```

Click rates are compared **per unit of examination** (clicks / Σ examine[slot], with the position curve fitted in H2), so a
feature is not credited for the slot the old carousel happened to put it in. Results summary: `docs/FORENSICS.md`.

In [1]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from rtrec.data.load import load_history, load_live
from rtrec.data import forensics as FZ
pd.set_option("display.width", 160, "display.max_columns", 20)

DATA = pathlib.Path(os.environ.get("RTREC_DATA", ROOT / "data"))
h = load_history(DATA)
live = load_live(DATA)
print("source:", h.source)
print(f"{len(h.catalog)} products, {len(h.sessions)} impressions, {len(h.events)} events;",
      "live tables:", {k: len(v) for k, v in live.items()})

source: {'source': 'synthetic', 'seed': 7, 'days': 60, 'small': False}
5000 products, 150457 impressions, 1664415 events; live tables: {'events': 412679, 'catalog_updates': 2989, 'requests': 21220, 'feedback': 21220}


In [2]:
fz = FZ.Forensics(h, max_sessions=400_000)   # samples impressions if the dump is bigger
def show(f):
    print(f.question); print("->", f.answer)
    return f.table

## H1
Is it worth showing what the shopper just viewed? Decides whether viewed items lead the list.

In [3]:
show(fz.h1_viewed_products())

H1 Does showing a product already viewed in the session convert?
-> Yes if > 1: a just-viewed product gets x18.09 clicks and x18.60 purchases per unit of examination vs other shown products (2151 pairs).


,pairs,clicks,purchases,exposure,ctr_per_exam,cvr_per_exam,p_buy_given_click
viewed,,,,,,,
False,1502419,3041,1760,503288.054619,0.006042,0.003497,0.578757
True,2151,79,47,722.735648,0.109307,0.065031,0.594937


## H2
Position curve: raw clicks per slot vs the fitted examination probability (product effects removed).

In [4]:
show(fz.h2_position_bias())

H2 Position bias curve (clicks per slot, corrected)
-> Slot 10 is examined 0.14x as much as slot 1 (fitted, product effects removed); the raw click curve says 0.12x. Top 3 slots carry 58% of the attention.


,impressions,clicks,purchases,mean_propensity,raw_ctr,raw_rel,fitted_examine
slot,,,,,,,
1,150457,984,593,0.022982,0.006540,1.000000,1.000000
2,150457,511,286,0.021387,0.003396,0.519309,0.532843
3,150457,397,238,0.019862,0.002639,0.403455,0.422996
4,150457,260,150,0.018636,0.001728,0.264228,0.282739
5,150457,235,139,0.017509,0.001562,0.238821,0.259567
6,150457,200,120,0.016166,0.001329,0.203252,0.226639
7,150457,159,86,0.015068,0.001057,0.161585,0.182560
8,150457,142,67,0.014208,0.000944,0.144309,0.165463
9,150457,117,68,0.013454,0.000778,0.118902,0.138649


## H3
Session similarity features: same subcategory / category / brand as the views, price ratio.

In [5]:
show(fz.h3_what_drives_clicks())

H3 What explains the click: same subcategory, category, brand, price close to what was viewed?
-> Strongest single signal: same_subcategory (x6.63 clicks per examination). By price ratio to the viewed products: <0.5: 0.0056, 0.5-0.8: 0.0073, 0.8-1.25: 0.0083, 1.25-2: 0.0076, >2: 0.0047 clicks/exam.


,share_of_pairs,click_lift,purchase_lift,pairs,ctr_per_exam,cvr_per_exam
feature,,,,,,
same_subcategory,0.025123,6.630502,6.544827,NaN,NaN,NaN
same_category,0.127986,3.349845,3.204702,NaN,NaN,NaN
same_brand,0.026517,4.656182,5.071642,NaN,NaN,NaN
price_ratio <0.5,NaN,NaN,NaN,420665.0,0.005614,0.003352
price_ratio 0.5-0.8,NaN,NaN,NaN,182055.0,0.007274,0.004556
price_ratio 0.8-1.25,NaN,NaN,NaN,166574.0,0.008323,0.004891
price_ratio 1.25-2,NaN,NaN,NaN,142241.0,0.007619,0.004219
price_ratio >2,NaN,NaN,NaN,217495.0,0.004716,0.002211


## H4
Price sensitivity by device and region (logit slope of P(buy | cart) on log price).

In [6]:
show(fz.h4_price_sensitivity())

H4 Does purchase-given-interest depend on price x device / region?
-> Logit slope of P(buy | cart) on log(price), by device: android -0.56, ios -0.17, web -0.38; across regions it ranges -0.62 .. +0.01. More negative = more price sensitive.


carts  p_buy_given_cart  log_price_slope
dim    value                                                          
device android                71737          0.371538        -0.564363
       ios                    85323          0.301818        -0.172314
       web                   127919          0.338402        -0.384872
region Andalucía              52288          0.346179        -0.471145
       Aragón                  8272          0.311775        -0.277297
       Asturias                6055          0.336416        -0.459288
       Baleares                6398          0.282119         0.011506
       Canarias               14710          0.327940        -0.430500
       Cantabria               2869          0.295573        -0.121143
       Castilla y León        13935          0.303983        -0.290239
       Castilla-La Mancha     14127          0.326325        -0.196464
       Catalunya              41321          0.313424        -0.203660
       Comunidad Valenciana   31610          0.353179        -0.458012
       Extremadura             6939          0.346448        -0.501951
       Galicia                16134          0.379509        -0.603873
       La Rioja                1828          0.385667        -0.621984
       Madrid                 40493          0.349789        -0.373605
       Murcia                  8852          0.316878        -0.348592
       Navarra                 5164          0.315453        -0.299222
       País Vasco             13984          0.336670        -0.472137

## H5
Long-term history: earlier-session affinity, and the repeat-purchase penalty.

In [7]:
show(fz.h5_user_history())

H5 Does long-term history predict? How much does having bought it already hurt?
-> Already bought: x0.28 clicks per examination (9395 pairs). Subcategory seen in an earlier session: x1.39. 75% of impressions are by shoppers already active in the first week of history.


pairs  clicks  purchases       exposure  ctr_per_exam  cvr_per_exam  p_buy_given_click
feature        bought_before                                                                                          
bought_before  False          1495175    3114       1804  500611.968157      0.006220      0.003604           0.579319
               True              9395       6          3    3398.822110      0.001765      0.000883           0.500000
known_affinity False          1191149    2281       1328  398681.735801      0.005721      0.003331           0.582201
               True            313421     839        479  105329.054466      0.007966      0.004548           0.570918

## H6
Do subcategory view surges in organic events predict carousel clicks, and how long does it last?

In [8]:
show(fz.h6_trends())

H6 Do trends in the organic events predict carousel conversion? With what delay?
-> Products whose subcategory is in the top 10% of 15-min view lift get x1.15 clicks per examination right away; the signal is strongest at lag 0 min and x0.90 after 120 min (decay = half-life of the counters).


,hot_threshold_lift,click_lift_hot_vs_rest,purchase_lift_hot_vs_rest
lag_min,,,
0,3.000000,1.145743,1.176683
15,3.000000,1.055205,1.126453
30,3.000000,0.882920,0.921524
60,3.157895,0.980050,0.937333
120,3.428571,0.903207,0.948758


## H7
Price drops and raises (needs live catalog_updates).

In [9]:
show(fz.h7_price_changes(live.get('events'), live.get('catalog_updates')))

H7 Effect of a price change
-> 0.0% of products change price within the history. Live: purchases per view x1.52 after a price drop, x0.95 after a raise.


event_type   add_to_cart  purchase   view  buy_per_view
kind  after                                            
drop  False         4876      1566  19902      0.078686
      True          7089      2637  22046      0.119614
raise False         3012      1026  10871      0.094380
      True          3419      1170  13075      0.089484

## H8
Shape of the target: one click / purchase per impression, purchase implies click.

In [10]:
show(fz.h8_target_shape())

H8 At most one click / purchase per impression? Does purchase imply click?
-> One clicked and one purchased field per impression; purchased == clicked in 100% of purchases; 2.07% of impressions get a click, 1.20% a purchase (58% of clicks convert).


,impressions,click_rate,purchase_rate,p_buy_given_click,purchase_equals_click,purchase_without_click,revenue_per_impression
0,150457,0.020737,0.01201,0.579167,1.0,0,0.157068


## H9
Structure of the logged propensities (decides the off-policy estimator).

In [11]:
show(fz.h9_propensity())

H9 Structure of `propensity`
-> Consistent with a Plackett-Luce draw over a per-day weight vector (slot-k propensity predicted from slot-1 propensities with median error 0.0%), median propensity 0.0008, 52% below 1e-3 (IPS weights > 1000: clip or use SNIPS / per-slot estimators).


,min,p01,median,p99,share_below_1e-3,pl_check_pairs,pl_median_rel_error,slot1_top_product_propensity
0,0.000059,0.000063,0.000811,0.133858,0.518581,430902,0.000005,0.126787


## H10, H11 (live tables)
Requests mix and stock-out risk; need `data/live/` (download with `--live`, or `sim.dump --live-minutes`).

In [12]:
if "requests" in live:
    display(show(FZ.h10_live_requests(live["requests"])))
if "catalog_updates" in live:
    top = h.events[h.events.event_type == "purchase"].product_id.value_counts().index[:100].tolist()
    display(show(FZ.h11_stock_risk(live["catalog_updates"], top, live.get("feedback"))))

H10 Live requests: rate, % new shoppers, % without views, repetition
-> 2.95 requests/s, 50% new shoppers, 25% without recent views (mean 2.3 views); pages product 45%, home 29%, category 26%. Visibility delay of requests is measured by the runtime (latency_ms in feedback), not here.


,requests,per_second,new_user_share,no_view_share,mean_views,requests_per_user,users_seen_twice_share
0,21220,2.947447,0.49722,0.250236,2.256032,1.258003,0.197415


H11 Products going out of stock right after we answer
-> 396 stock-outs/hour (30 among the top 100); a 10-item list of top products goes invalid within 3s with p = 0.25% -> refresh stock <= 0.5 s before sending. Measured in feedback: 0.00% invalid.


,oos_per_hour,oos_per_hour_in_top,p_list_invalid_within_horizon,feedback_invalid_share
0,396.196646,30.052843,0.002501,0.0
